# 04 - Model Training
## Навчання моделей з GridSearch (2d)

У цьому notebook ми:
- Завантажимо ембединги
- Розділимо дані на train/test
- Навчимо кілька моделей з GridSearch
- Збережемо найкращі моделі

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

from src.embeddings import load_embeddings
from src.train import (
    split_data,
    train_model_with_gridsearch,
    save_model
)
from src.config import PROCESSED_DATA_PATH, MODELS_PATH, RANDOM_STATE

## 1. Завантаження даних

In [2]:
# Завантажуємо дані та ембединги
df = pd.read_csv(PROCESSED_DATA_PATH / 'cleaned_data.csv')
embeddings = load_embeddings(str(PROCESSED_DATA_PATH / 'embeddings.npy'))

print(f"Завантажено {len(df)} записів")
print(f"Розмір ембедингів: {embeddings.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\embeddings.npy'

In [ ]:
# Підготуємо features та labels
X = embeddings
y = df['label'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Унікальні мітки: {np.unique(y)}")

## 2. Розділення на train/test

In [ ]:
# Розділимо дані
X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")
print(f"\nTrain labels distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nTest labels distribution:")
print(pd.Series(y_test).value_counts())

## 3. Навчання моделі: Logistic Regression

In [ ]:
# Навчимо Logistic Regression
print("=" * 60)
print("Навчання Logistic Regression...")
print("=" * 60)

lr_model = train_model_with_gridsearch(
    X_train,
    y_train,
    model_name='logistic_regression',
    cv=5
)

# Оцінимо на тестових даних
train_score = lr_model.score(X_train, y_train)
test_score = lr_model.score(X_test, y_test)

print(f"\nТренувальна точність: {train_score:.4f}")
print(f"Тестова точність: {test_score:.4f}")

In [ ]:
# Збережемо модель
save_model(lr_model, 'logistic_regression')

## 4. Навчання моделі: Random Forest

In [ ]:
# Навчимо Random Forest
print("=" * 60)
print("Навчання Random Forest...")
print("=" * 60)

rf_model = train_model_with_gridsearch(
    X_train,
    y_train,
    model_name='random_forest',
    cv=5
)

# Оцінимо на тестових даних
train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)

print(f"\nТренувальна точність: {train_score:.4f}")
print(f"Тестова точність: {test_score:.4f}")

In [ ]:
# Збережемо модель
save_model(rf_model, 'random_forest')

## 5. Навчання моделі: SVM

In [ ]:
# Навчимо SVM (на підмножині даних для швидкості)
print("=" * 60)
print("Навчання SVM...")
print("=" * 60)

# Використаємо менше даних для SVM, якщо датасет великий
sample_size = min(5000, len(X_train))
indices = np.random.choice(len(X_train), sample_size, replace=False)
X_train_sample = X_train[indices]
y_train_sample = y_train[indices]

svm_model = train_model_with_gridsearch(
    X_train_sample,
    y_train_sample,
    model_name='svm',
    cv=3
)

# Оцінимо на тестових даних
train_score = svm_model.score(X_train, y_train)
test_score = svm_model.score(X_test, y_test)

print(f"\nТренувальна точність: {train_score:.4f}")
print(f"Тестова точність: {test_score:.4f}")

In [ ]:
# Збережемо модель
save_model(svm_model, 'svm')

## 6. Порівняння моделей

In [ ]:
# Порівняємо всі моделі
results = {
    'Logistic Regression': {
        'train': lr_model.score(X_train, y_train),
        'test': lr_model.score(X_test, y_test)
    },
    'Random Forest': {
        'train': rf_model.score(X_train, y_train),
        'test': rf_model.score(X_test, y_test)
    },
    'SVM': {
        'train': svm_model.score(X_train, y_train),
        'test': svm_model.score(X_test, y_test)
    }
}

results_df = pd.DataFrame(results).T
print("\nПорівняння моделей:")
print(results_df)

In [ ]:
# Візуалізуємо результати
import matplotlib.pyplot as plt

results_df.plot(kind='bar', figsize=(10, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.xticks(rotation=45)
plt.legend(['Train', 'Test'])
plt.tight_layout()
plt.show()

## 7. Збереження результатів навчання

In [ ]:
# Збережемо train/test індекси для відтворюваності
np.save(PROCESSED_DATA_PATH / 'train_indices.npy', 
        np.arange(len(X_train)))
np.save(PROCESSED_DATA_PATH / 'test_indices.npy', 
        np.arange(len(X_train), len(X_train) + len(X_test)))

print("Збережено індекси train/test")

## Висновки

✅ Навчено три моделі з GridSearch оптимізацією
✅ Знайдено найкращі гіперпараметри для кожної моделі
✅ Оцінено продуктивність на тестових даних
✅ Збережено моделі для подальшого використання